# P7-3 · Video và embodied AI (nâng cao)

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import numpy as np
def sample_frames(n_frames, k, train, rng=np.random.default_rng(0)):
    edges = np.linspace(0, n_frames, k + 1)                     # k đoạn bằng nhau
    if train:
        return [int(rng.uniform(edges[i], edges[i + 1])) for i in range(k)]
    return [int((edges[i] + edges[i + 1]) / 2) for i in range(k)]

sample_frames(150, 16, train=False)   # [4, 14, 23, 32, 42, 51, 60, 70, 79, 89, 98, 107, 117, 126, 135, 145]

[4, 14, 23, 32, 42, 51, 60, 70, 79, 89, 98, 107, 117, 126, 135, 145]


## Bài tập 5: Đặc trưng khung hình + Transformer thời gian {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn

class Anticipator(nn.Module):
    def __init__(self, n_verbs, n_nouns, d_in=512, d=256, T=8, layers=2):
        super().__init__()
        self.proj = nn.Linear(d_in, d)
        self.pos = nn.Parameter(torch.zeros(1, T + 1, d))
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        enc = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=4 * d, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.verb, self.noun = nn.Linear(d, n_verbs), nn.Linear(d, n_nouns)

    def forward(self, feats):                       # feats: (B, T, d_in)
        x = self.proj(feats)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1) + self.pos
        h = self.encoder(x)[:, 0]
        return self.verb(h), self.noun(h)

m = Anticipator(n_verbs=97, n_nouns=300)
v, n = m(torch.randn(4, 8, 512))
print(v.shape, n.shape)

torch.Size([4, 97]) torch.Size([4, 300])
